# Xây dựng đồ thị

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/HuyBui2112/detecting_variable_misuse_bugs.git'
BRANCH = 'main'
REPO_DIR = Path('/content/detecting_variable_misuse_bugs')

DRIVE_WORK_DIR = Path('/content/drive/MyDrive/variable_misuse_graph_build')
DRIVE_PART_DIR = DRIVE_WORK_DIR / 'dataset_parts'
RUNTIME_PART_DIR = REPO_DIR / 'dataset_parts'

SOURCE_PROCESSED_ROOT = DRIVE_WORK_DIR / 'dataset' / 'processed' / 'great_colab'

# Chọn kích thước subset ở đây. 100_000 nhẹ hơn; 300_000 hợp lý cho đồ án.
SUBSET_TOTAL_SAMPLES = 300_000
SUBSET_SEED = 42
SUBSET_PROCESSED_ROOT = (
    DRIVE_WORK_DIR / 'dataset' / 'processed' / f'great_colab_balanced_{SUBSET_TOTAL_SAMPLES}'
)

GRAPH_OUTPUT_ROOT = (
    DRIVE_WORK_DIR / 'dataset' / 'graphs' / f'great_colab_balanced_{SUBSET_TOTAL_SAMPLES}'
)

# Ba variant bắt buộc để so sánh AST-only với Data Flow.
GRAPH_VARIANTS_TO_BUILD = (
    'ast_only',
    'ast_next_token',
    'ast_next_token_data_flow',
)

DRIVE_PART_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('REPO_DIR:', REPO_DIR)
print('DRIVE_PART_DIR:', DRIVE_PART_DIR)
print('RUNTIME_PART_DIR:', RUNTIME_PART_DIR)
print('SOURCE_PROCESSED_ROOT:', SOURCE_PROCESSED_ROOT)
print('SUBSET_PROCESSED_ROOT:', SUBSET_PROCESSED_ROOT)
print('GRAPH_OUTPUT_ROOT:', GRAPH_OUTPUT_ROOT)
print('GRAPH_VARIANTS_TO_BUILD:', GRAPH_VARIANTS_TO_BUILD)

REPO_DIR: /content/detecting_variable_misuse_bugs
DRIVE_PART_DIR: /content/drive/MyDrive/variable_misuse_graph_build/dataset_parts
RUNTIME_PART_DIR: /content/detecting_variable_misuse_bugs/dataset_parts
SOURCE_PROCESSED_ROOT: /content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab
SUBSET_PROCESSED_ROOT: /content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000
GRAPH_OUTPUT_ROOT: /content/drive/MyDrive/variable_misuse_graph_build/dataset/graphs/great_colab_balanced_300000
GRAPH_VARIANTS_TO_BUILD: ('ast_only', 'ast_next_token', 'ast_next_token_data_flow')


In [3]:
!python --version
!python -m pip --version
!python -c "import platform; print(platform.platform())"
!df -h /content /content/drive || true
!nvidia-smi || true

Python 3.12.13
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Linux-6.6.122+-x86_64-with-glibc2.35
df: /content/drive: No such file or directory
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /
/bin/bash: line 1: nvidia-smi: command not found


In [4]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))

print('Current dir:', os.getcwd())
!git log -1 --oneline
!git status --short

Current dir: /content/detecting_variable_misuse_bugs
e62d519 (HEAD -> main, origin/main, origin/HEAD) create Layout Build Graph


In [5]:
from variable_misuse_gnn.graph.config import GraphConstructionConfig
from variable_misuse_gnn.graph.pipeline import run_graph_construction
from variable_misuse_gnn.graph.variants import GRAPH_VARIANTS

print('Graph variants supported:', sorted(GRAPH_VARIANTS))

Graph variants supported: ['ast_next_token', 'ast_next_token_data_flow', 'ast_only', 'full_available']


In [6]:
import json

source_summary = json.loads((SOURCE_PROCESSED_ROOT / 'preprocessing_summary.json').read_text(encoding='utf-8'))
print(json.dumps(source_summary.get('aggregate', {}), ensure_ascii=False, indent=2)[:3000])

for split in ('train', 'dev', 'eval'):
    split_path = SOURCE_PROCESSED_ROOT / f'{split}.jsonl'
    with split_path.open(encoding='utf-8') as handle:
        sample = json.loads(handle.readline())
    print(split, sorted(sample.keys()), 'tokens=', len(sample['tokens']), 'edges=', len(sample['edges']))

{
  "counters": {
    "read": 2952990,
    "written": 2884323,
    "dropped_too_many_tokens": 53068,
    "fixed_repair_target_added_to_candidates": 12666,
    "dropped_bug_sample_without_repair_target": 15598,
    "dropped_too_few_repair_candidates": 1
  },
  "class_counts": {
    "bug": 1434367,
    "no_bug": 1449956
  },
  "edge_group_counts": {
    "syntax": 79930821,
    "next_token": 554258762,
    "lexical": 36833980,
    "control_flow": 25268818,
    "data_flow": 81309475,
    "semantic": 130614
  },
  "license_counts": {
    "bsd-3-clause": 783565,
    "mit": 840222,
    "apache-2.0": 1056276,
    "bsd-2-clause": 161362,
    "isc": 12757,
    "gpl-3.0": 16356,
    "mpl-2.0": 3155,
    "cc0-1.0": 2047,
    "gpl-2.0": 6729,
    "epl-1.0": 726,
    "lgpl-3.0": 397,
    "lgpl-2.1": 593,
    "unlicense": 138
  }
}
train ['candidates', 'edges', 'has_bug', 'id', 'loc', 'provenance', 'split', 'targets', 'tokens'] tokens= 34 edges= 95
dev ['candidates', 'edges', 'has_bug', 'id', 'loc', 

In [7]:
import random

SPLITS = ('train', 'dev', 'eval')
SPLIT_SEEDS = {'train': 101, 'dev': 202, 'eval': 303}

def split_written_counts(summary: dict) -> dict[str, int]:
    """Đọc số sample đã tiền xử lý của từng split từ summary gốc."""
    counts = {}
    for split_report in summary['splits']:
        counts[split_report['split']] = int(split_report['counters']['written'])
    return counts

split_counts = split_written_counts(source_summary)
total_written = sum(split_counts.values())
split_quotas = {
    split: round(SUBSET_TOTAL_SAMPLES * split_counts[split] / total_written)
    for split in SPLITS
}

# Bù sai số làm tròn vào train để tổng đúng SUBSET_TOTAL_SAMPLES.
split_quotas['train'] += SUBSET_TOTAL_SAMPLES - sum(split_quotas.values())
class_quotas = {
    split: {'bug': quota // 2, 'no_bug': quota - quota // 2}
    for split, quota in split_quotas.items()
}

print('Split counts gốc:', split_counts)
print('Split quotas subset:', split_quotas)
print('Class quotas:', class_quotas)
print('Subset output:', SUBSET_PROCESSED_ROOT)

Split counts gốc: {'train': 1756340, 'dev': 181478, 'eval': 946505}
Split quotas subset: {'train': 182678, 'dev': 18876, 'eval': 98446}
Class quotas: {'train': {'bug': 91339, 'no_bug': 91339}, 'dev': {'bug': 9438, 'no_bug': 9438}, 'eval': {'bug': 49223, 'no_bug': 49223}}
Subset output: /content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000


In [8]:
def reservoir_sample_ids(input_path: Path, bug_quota: int, no_bug_quota: int, seed: int) -> tuple[set[str], dict]:
    """Chọn sample id cân bằng nhãn bằng reservoir sampling."""
    rng = random.Random(seed)
    reservoirs = {'bug': [], 'no_bug': []}
    seen = {'bug': 0, 'no_bug': 0}
    quotas = {'bug': bug_quota, 'no_bug': no_bug_quota}

    with input_path.open('r', encoding='utf-8') as file:
        for line in file:
            item = json.loads(line)
            label = 'bug' if item['has_bug'] else 'no_bug'
            seen[label] += 1
            quota = quotas[label]
            if quota <= 0:
                continue
            reservoir = reservoirs[label]
            sample_id = item['id']
            if len(reservoir) < quota:
                reservoir.append(sample_id)
            else:
                index = rng.randint(0, seen[label] - 1)
                if index < quota:
                    reservoir[index] = sample_id

    selected_ids = set(reservoirs['bug']) | set(reservoirs['no_bug'])
    stats = {
        'seen': seen,
        'selected': {
            'bug': len(reservoirs['bug']),
            'no_bug': len(reservoirs['no_bug']),
            'total': len(selected_ids),
        },
    }
    return selected_ids, stats

def subset_ready(root: Path) -> bool:
    """Kiểm tra subset đã có đủ file cần thiết chưa."""
    return all((root / f'{split}.jsonl').exists() for split in SPLITS) and (root / 'preprocessing_summary.json').exists()

if subset_ready(SUBSET_PROCESSED_ROOT):
    print('Subset đã tồn tại, dùng lại:', SUBSET_PROCESSED_ROOT)
else:
    SUBSET_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
    subset_summary = {
        'source_root': SOURCE_PROCESSED_ROOT.as_posix(),
        'output_root': SUBSET_PROCESSED_ROOT.as_posix(),
        'subset_total_samples': SUBSET_TOTAL_SAMPLES,
        'seed': SUBSET_SEED,
        'split_quotas': split_quotas,
        'class_quotas': class_quotas,
        'splits': [],
    }

    for split in SPLITS:
        input_path = SOURCE_PROCESSED_ROOT / f'{split}.jsonl'
        output_path = SUBSET_PROCESSED_ROOT / f'{split}.jsonl'
        quotas = class_quotas[split]

        selected_ids, sampling_stats = reservoir_sample_ids(
            input_path=input_path,
            bug_quota=quotas['bug'],
            no_bug_quota=quotas['no_bug'],
            seed=SUBSET_SEED + SPLIT_SEEDS[split],
        )

        written = 0
        class_counts = {'bug': 0, 'no_bug': 0}
        with input_path.open('r', encoding='utf-8') as input_file, output_path.open('w', encoding='utf-8') as output_file:
            for line in input_file:
                item = json.loads(line)
                if item['id'] not in selected_ids:
                    continue
                output_file.write(line)
                written += 1
                label = 'bug' if item['has_bug'] else 'no_bug'
                class_counts[label] += 1

        split_report = {
            'split': split,
            'input_path': input_path.as_posix(),
            'output_path': output_path.as_posix(),
            'quota': split_quotas[split],
            'class_quota': quotas,
            'sampling_stats': sampling_stats,
            'written': written,
            'class_counts': class_counts,
        }
        subset_summary['splits'].append(split_report)
        print(split_report)

    (SUBSET_PROCESSED_ROOT / 'preprocessing_summary.json').write_text(
        json.dumps(subset_summary, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print('Đã tạo subset:', SUBSET_PROCESSED_ROOT)

{'split': 'train', 'input_path': '/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab/train.jsonl', 'output_path': '/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/train.jsonl', 'quota': 182678, 'class_quota': {'bug': 91339, 'no_bug': 91339}, 'sampling_stats': {'seen': {'bug': 873283, 'no_bug': 883057}, 'selected': {'bug': 91339, 'no_bug': 91339, 'total': 182678}}, 'written': 182678, 'class_counts': {'bug': 91339, 'no_bug': 91339}}
{'split': 'dev', 'input_path': '/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab/dev.jsonl', 'output_path': '/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/dev.jsonl', 'quota': 18876, 'class_quota': {'bug': 9438, 'no_bug': 9438}, 'sampling_stats': {'seen': {'bug': 90266, 'no_bug': 91212}, 'selected': {'bug': 9438, 'no_bug': 9438, 'total': 18876}}, 'written': 18876, 'class_counts': {'bug': 9438, 'no_bu

In [9]:
subset_summary = json.loads((SUBSET_PROCESSED_ROOT / 'preprocessing_summary.json').read_text(encoding='utf-8'))
print(json.dumps(subset_summary, ensure_ascii=False, indent=2)[:4000])

for split in SPLITS:
    path = SUBSET_PROCESSED_ROOT / f'{split}.jsonl'
    with path.open(encoding='utf-8') as handle:
        first = json.loads(handle.readline())
    print(split, path, 'first keys=', sorted(first.keys()))

{
  "source_root": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab",
  "output_root": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000",
  "subset_total_samples": 300000,
  "seed": 42,
  "split_quotas": {
    "train": 182678,
    "dev": 18876,
    "eval": 98446
  },
  "class_quotas": {
    "train": {
      "bug": 91339,
      "no_bug": 91339
    },
    "dev": {
      "bug": 9438,
      "no_bug": 9438
    },
    "eval": {
      "bug": 49223,
      "no_bug": 49223
    }
  },
  "splits": [
    {
      "split": "train",
      "input_path": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab/train.jsonl",
      "output_path": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/train.jsonl",
      "quota": 182678,
      "class_quota": {
        "bug": 91339,
        "no_bug": 91339
      },
      "sampling_stats": {
        "seen": {
 

In [10]:
subset_summary = json.loads((SUBSET_PROCESSED_ROOT / 'preprocessing_summary.json').read_text(encoding='utf-8'))
print(json.dumps(subset_summary, ensure_ascii=False, indent=2)[:4000])

for split in SPLITS:
    path = SUBSET_PROCESSED_ROOT / f'{split}.jsonl'
    with path.open(encoding='utf-8') as handle:
        first = json.loads(handle.readline())
    print(split, path, 'first keys=', sorted(first.keys()))

{
  "source_root": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab",
  "output_root": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000",
  "subset_total_samples": 300000,
  "seed": 42,
  "split_quotas": {
    "train": 182678,
    "dev": 18876,
    "eval": 98446
  },
  "class_quotas": {
    "train": {
      "bug": 91339,
      "no_bug": 91339
    },
    "dev": {
      "bug": 9438,
      "no_bug": 9438
    },
    "eval": {
      "bug": 49223,
      "no_bug": 49223
    }
  },
  "splits": [
    {
      "split": "train",
      "input_path": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab/train.jsonl",
      "output_path": "/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/train.jsonl",
      "quota": 182678,
      "class_quota": {
        "bug": 91339,
        "no_bug": 91339
      },
      "sampling_stats": {
        "seen": {
 

In [11]:
dry_config = GraphConstructionConfig(
    input_root=SUBSET_PROCESSED_ROOT,
    output_root=GRAPH_OUTPUT_ROOT,
    graph_variant='ast_next_token_data_flow',
    splits=SPLITS,
    max_samples_per_split=1000,
    dry_run=True,
)

dry_report = run_graph_construction(dry_config)
print(json.dumps(dry_report['aggregate'], ensure_ascii=False, indent=2))

{
  "counters": {
    "read": 3000,
    "written": 3000
  },
  "class_counts": {
    "bug": 1550,
    "no_bug": 1450
  },
  "edge_type_counts": {
    "syntax": 81068,
    "syntax_reverse": 81068,
    "next_token": 569221,
    "next_token_reverse": 569221,
    "lexical": 37845,
    "lexical_reverse": 37845,
    "data_flow": 82585,
    "data_flow_reverse": 82585
  }
}


In [12]:
variant_reports = {}

for graph_variant in GRAPH_VARIANTS_TO_BUILD:
    print('\n=== Build variant:', graph_variant, '===')
    config = GraphConstructionConfig(
        input_root=SUBSET_PROCESSED_ROOT,
        output_root=GRAPH_OUTPUT_ROOT,
        graph_variant=graph_variant,
        splits=SPLITS,
        max_samples_per_split=None,
        shard_size=50000,
        dry_run=False,
    )
    report = run_graph_construction(config)
    variant_reports[graph_variant] = report['aggregate']
    print(json.dumps(report['aggregate'], ensure_ascii=False, indent=2))

print('\nDone. Variant reports:')
print(json.dumps(variant_reports, ensure_ascii=False, indent=2))


=== Build variant: ast_only ===
{
  "counters": {
    "read": 300000,
    "written": 300000
  },
  "class_counts": {
    "bug": 150000,
    "no_bug": 150000
  },
  "edge_type_counts": {
    "syntax": 8315620,
    "syntax_reverse": 8315620
  }
}

=== Build variant: ast_next_token ===
{
  "counters": {
    "read": 300000,
    "written": 300000
  },
  "class_counts": {
    "bug": 150000,
    "no_bug": 150000
  },
  "edge_type_counts": {
    "syntax": 8315620,
    "syntax_reverse": 8315620,
    "next_token": 57649150,
    "next_token_reverse": 57649150
  }
}

=== Build variant: ast_next_token_data_flow ===
{
  "counters": {
    "read": 300000,
    "written": 300000
  },
  "class_counts": {
    "bug": 150000,
    "no_bug": 150000
  },
  "edge_type_counts": {
    "syntax": 8315620,
    "syntax_reverse": 8315620,
    "next_token": 57649150,
    "next_token_reverse": 57649150,
    "lexical": 3830171,
    "lexical_reverse": 3830171,
    "data_flow": 8452509,
    "data_flow_reverse": 8452509
  

In [13]:
import json
from pathlib import Path

SAMPLE_EXPORT_ROOT = DRIVE_WORK_DIR / "samples_for_embedding_layout"
SAMPLE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

def export_graph_samples(variant: str, split: str, max_samples: int = 10):
    graph_split_dir = GRAPH_OUTPUT_ROOT / variant / split
    output_path = SAMPLE_EXPORT_ROOT / f"{variant}_{split}_samples.jsonl"

    written = 0
    with output_path.open("w", encoding="utf-8") as output_file:
        for shard_path in sorted(graph_split_dir.glob("shard_*.jsonl")):
            with shard_path.open("r", encoding="utf-8") as input_file:
                for line in input_file:
                    item = json.loads(line)
                    output_file.write(json.dumps(item, ensure_ascii=False) + "\n")
                    written += 1
                    if written >= max_samples:
                        print("Wrote", written, "samples to", output_path)
                        return output_path

    print("Wrote", written, "samples to", output_path)
    return output_path

for variant in ["ast_only", "ast_next_token", "ast_next_token_data_flow"]:
    for split in ["train", "dev"]:
        export_graph_samples(variant, split, max_samples=10)

Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_only_train_samples.jsonl
Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_only_dev_samples.jsonl
Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_train_samples.jsonl
Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_dev_samples.jsonl
Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_data_flow_train_samples.jsonl
Wrote 10 samples to /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_data_flow_dev_samples.jsonl


In [14]:
import shutil

for variant in ["ast_only", "ast_next_token", "ast_next_token_data_flow"]:
    src = GRAPH_OUTPUT_ROOT / variant / "graph_construction_summary.json"
    dst = SAMPLE_EXPORT_ROOT / f"{variant}_graph_construction_summary.json"
    if src.exists():
        shutil.copy2(src, dst)
        print("Copied:", dst)
    else:
        print("Missing:", src)

Copied: /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_only_graph_construction_summary.json
Copied: /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_graph_construction_summary.json
Copied: /content/drive/MyDrive/variable_misuse_graph_build/samples_for_embedding_layout/ast_next_token_data_flow_graph_construction_summary.json
